# 实验报告 03：正则表达式与网络结构化数据

| 项目 | 内容 |
| --- | --- |
| 学号 | 20244080130 |
| 姓名 | 廖晨宇 |
| 课程实验 | 实验 03 |
| Conda 环境 | `exp03`（独立虚拟环境，非 base） |
| Python 版本 | 3.12 |
| 运行方式 | 重启内核 → Run All，全部代码单元按顺序执行、无报错 |

> 提交文件：`20244080130_廖晨宇_exp03.ipynb`

## 一、实验环境与提交要求

1. 本实验使用独立的 conda 虚拟环境 **`exp03`**（Python 3.12），未使用 base 环境；环境名由第一个代码单元通过 `CONDA_DEFAULT_ENV` 打印出来验证。
2. 第一个代码单元填写学号并记录机器指纹，运行后得到本次实验的 `RUN_ID`。
3. Markdown 单元解释匹配规则与解析结果，代码单元完成计算，每个任务都打印关键输出。
4. 提交前重启内核并 Run All，确认没有依赖旧变量的残留结果。

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080130
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] exp03
[学号]       20244080130
[机器指纹] 01534316381bc475
[RUN_ID]   20244080130-01534316381bc475
[Python]    3.12.16


## 二、实验内容与过程

### 实验1：正则表达式的基础匹配

正则表达式的元字符各有分工：

- `\d` 匹配一个数字字符（`\d+` 即一个或多个连续数字）；
- `\s` 匹配一个空白字符（空格、制表符、换行等，`\s+` 匹配一整串空白）；
- `\b` 是**单词边界**（zero-width 断言），本身不占用字符，只在“单词字符（字母、数字、下划线）”与“非单词字符/字符串端点”交界处成立。

下面在 `"cat on mat; later: cats and dogs"` 上分别测试这三条正则。

In [2]:
import re

text1 = "cat on mat; later: cats and dogs"
print("原文：", repr(text1))
print("-" * 64)

m_digits = re.findall(r"\d+", text1)
m_spaces = re.findall(r"\s+", text1)
m_cat = re.findall(r"\bcat\b", text1)

print(r"1) \d+     ->", m_digits, f"（共 {len(m_digits)} 个匹配）")
print(r"2) \s+     ->", m_spaces, f"（共 {len(m_spaces)} 个匹配）")
print(r"3) \bcat\b ->", m_cat, f"（共 {len(m_cat)} 个匹配）")

print("\n\\bcat\\b 各次匹配的位置：")
for m in re.finditer(r"\bcat\b", text1):
    print(f"  匹配 {m.group()!r}  span={m.span()}")

原文： 'cat on mat; later: cats and dogs'
----------------------------------------------------------------
1) \d+     -> [] （共 0 个匹配）
2) \s+     -> [' ', ' ', ' ', ' ', ' ', ' '] （共 6 个匹配）
3) \bcat\b -> ['cat'] （共 1 个匹配）

\bcat\b 各次匹配的位置：
  匹配 'cat'  span=(0, 3)


**匹配规则解释（任务 1）**

| 正则 | 结果 | 原因 |
| --- | --- | --- |
| `\d+` | `[]`（0 个） | 原句中不含任何数字字符，`\d` 无处可匹配 |
| `\s+` | 6 个单空格 `' '` | 句中 7 个单词之间有 6 个空格；`\s+` 是贪婪的，遇到相邻空白会一次吞掉一段，所以每个匹配恰好是一个空格 |
| `\bcat\b` | `['cat']`（1 个） | 只匹配独立的 `cat`。`cats` 虽然有 `cat` 前缀，但 `t` 与 `s` 之间两边都是单词字符，不构成单词边界，因此 `cats` 不匹配 |

结论：`\b` 保证的是“完整的单词”，而不是“单词的前缀”。若只想匹配单词内部，可以用 `\bcat`（不加尾部 `\b`），那样 `cats` 也会被匹配到。

**任务 2：交替符 `|` 与分支顺序**

交替符按**书写顺序从左到右尝试**，一旦某个分支在当前起始位置匹配成功就立即采用，`findall` 再从匹配终点继续往后扫描，因此“先写长的分支”与“先写短的分支”结果完全不同。

In [3]:
text2 = "cat dog bird CAT"
print("原文：", repr(text2))
print(r"re.findall(r'cat|dog', text2)      ->", re.findall(r"cat|dog", text2))
print(r"re.findall(r'cat|dog', text2, re.I) ->", re.findall(r"cat|dog", text2, re.IGNORECASE),
      "（加 re.IGNORECASE 后 CAT 也能匹配）")

print("\n同一字符串 \"aaaa\" 上的分支顺序差异：")
a1 = re.findall(r"a|aa|aaa", "aaaa")
a2 = re.findall(r"aaa|aa|a", "aaaa")
print(r'  re.findall(r"a|aa|aaa", "aaaa") ->', a1, f"共 {len(a1)} 处匹配")
print(r'  re.findall(r"aaa|aa|a", "aaaa") ->', a2, f"共 {len(a2)} 处匹配")

原文： 'cat dog bird CAT'
re.findall(r'cat|dog', text2)      -> ['cat', 'dog']
re.findall(r'cat|dog', text2, re.I) -> ['cat', 'dog', 'CAT'] （加 re.IGNORECASE 后 CAT 也能匹配）

同一字符串 "aaaa" 上的分支顺序差异：
  re.findall(r"a|aa|aaa", "aaaa") -> ['a', 'a', 'a', 'a'] 共 4 处匹配
  re.findall(r"aaa|aa|a", "aaaa") -> ['aaa', 'a'] 共 2 处匹配


**结果解释（任务 2）**

- `cat|dog` 在 `"cat dog bird CAT"` 上得到 `['cat', 'dog']`：正则默认区分大小写，`CAT` 不会被匹配；加 `re.IGNORECASE` 后变成 `['cat', 'dog', 'CAT']`。
- `a|aa|aaa` 得到 `['a', 'a', 'a', 'a']`：每个位置先试最短分支 `a` 就成功，于是把 `aaaa` 切成 4 个单字母，共 4 处匹配。
- `aaa|aa|a` 得到 `['aaa', 'a']`：第一个位置先试 `aaa` 成功，消耗掉 3 个 a，剩下 1 个 a 再由最后的分支 `a` 吃掉，共 2 处匹配。

要点：交替是**有序的、非贪婪地按书写顺序选择**（不是“最长优先”），所以写正则时要把更具体、更长的分支放在前面。

**任务 3：量词对 `a` 个数的约束**

在 `"ct cat caat caaat"` 上依次测试 `ca*t`、`ca+t`、`ca?t`、`ca{2}t`、`ca{1,2}t`。

In [4]:
text3 = "ct cat caat caaat"
print("原文：", repr(text3))
print("-" * 64)

patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]
for p in patterns:
    print(f"{p:10s} -> {re.findall(p, text3)}")

原文： 'ct cat caat caaat'
----------------------------------------------------------------
ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


**量词含义与结果解释（任务 3）**

| 量词 | 含义 | 本文结果 | 说明 |
| --- | --- | --- | --- |
| `*` | 前面元素出现 **0 次或多次** | `['ct', 'cat', 'caat', 'caaat']` | 允许 0 个 `a`，所以连 `ct` 也算匹配 |
| `+` | **1 次或多次** | `['cat', 'caat', 'caaat']` | 至少要 1 个 `a`，`ct` 被排除 |
| `?` | **0 次或 1 次** | `['ct', 'cat']` | 最多只能夹 1 个 `a`；`caat`/`caaat` 中间有 2~3 个 `a`，无匹配 |
| `{2}` | **恰好 2 次** | `['caat']` | 只有 `caat` 正好夹 2 个 `a` |
| `{1,2}` | **1 到 2 次** | `['cat', 'caat']` | 1 个或 2 个 `a` 都可以，3 个 `a` 的 `caaat` 超出上限 |

补充：量词默认是**贪婪**的（`*`、`+`、`{m,n}` 会尽量多吞），配合 `?` 可变成懒惰匹配（如 `ca*?t`）；`findall` 返回的是互不重叠的匹配，所以同一段文本不会被重复计入。

### 实验2：分组、回溯引用与替换

**任务 1：用捕获组拆分邮箱的用户名与域名**

- `([\w.+-]+)` 是第 1 个捕获组：用户名，由单词字符、点、加号、减号组成（注意 `-` 放在字符类末尾避免被当成范围）；
- `@` 为字面量；
- `([\w-]+(?:\.[\w-]+)+)` 是第 2 个捕获组：域名，要求至少包含一个点，`(?:...)` 是非捕获组，只分组不额外产生编号。

In [5]:
text_mail = "联系：alice@example.com；备选：bob@test.org"
mail_pattern = r"([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)"

print("原文：", text_mail)
pairs = re.findall(mail_pattern, text_mail)
print(f"共找到 {len(pairs)} 个邮箱：\n")
for i, (user, domain) in enumerate(pairs, 1):
    print(f"  第 {i} 个：用户名 = {user!r}，域名 = {domain!r}")

print("\n用 finditer 查看分组编号：")
for m in re.finditer(mail_pattern, text_mail):
    print(f"  完整匹配 = {m.group(0)!r} | group(1) = {m.group(1)!r} | group(2) = {m.group(2)!r}")

原文： 联系：alice@example.com；备选：bob@test.org
共找到 2 个邮箱：

  第 1 个：用户名 = 'alice'，域名 = 'example.com'
  第 2 个：用户名 = 'bob'，域名 = 'test.org'

用 finditer 查看分组编号：
  完整匹配 = 'alice@example.com' | group(1) = 'alice' | group(2) = 'example.com'
  完整匹配 = 'bob@test.org' | group(1) = 'bob' | group(2) = 'test.org'


**结果解释（任务 1）**：`re.findall` 在正则含多个捕获组时返回**元组列表**，元组第 1 项对应第 1 个括号、第 2 项对应第 2 个括号，正好把两个邮箱拆成 `('alice', 'example.com')` 和 `('bob', 'test.org')`。中文标点 `：`、`；` 不属于 `\w`，因此天然起到分隔作用。

**任务 2：用后向引用找连续重复的单词**

- `\b(\w+)\b` 先捕获一个完整单词；
- `\s+` 匹配中间的空白；
- `\1` 是**后向引用**，表示“必须与第 1 个捕获组刚匹配到的内容完全相同”，从而实现重复检测。

In [6]:
text_dup = "cat cat, dog cat, bird bird"
dup_pattern = r"\b(\w+)\s+\1\b"

print("原文：", text_dup)
dups = re.findall(dup_pattern, text_dup)
print("连续重复的单词：", dups)

print("\n各重复片段的位置：")
for m in re.finditer(dup_pattern, text_dup):
    print(f"  片段 = {m.group(0)!r}，重复的单词 = {m.group(1)!r}，span = {m.span()}")

print("\n用后向引用把重复词压成一个：")
print("  ", re.sub(dup_pattern, r"\1", text_dup))

原文： cat cat, dog cat, bird bird
连续重复的单词： ['cat', 'bird']

各重复片段的位置：
  片段 = 'cat cat'，重复的单词 = 'cat'，span = (0, 7)
  片段 = 'bird bird'，重复的单词 = 'bird'，span = (18, 27)

用后向引用把重复词压成一个：
   cat, dog cat, bird


**结果解释（任务 2）**：只有 `cat cat` 和 `bird bird` 命中，得到 `['cat', 'bird']`；`dog cat` 虽然是相邻两个单词，但内容不同，`\1` 不成立，所以不算重复。替换表达式里 `\1` 又把捕获到的单词原样填回，于是 `cat cat` 被压成 `cat`。

**任务 3：日期格式转换（年-月-日 → 月-日-年）**

- `(\d{4})`：年，第 1 组；
- `([-/.])`：分隔符，第 2 组（分别记住实际用的是 `-`、`/` 还是 `.`）；
- `(\d{2})`：月，第 3 组；`(\d{2})`：日，第 4 组；
- `\2` 要求第二个分隔符与第一个**完全一致**，可避免把 `2024-05/01` 这类脏数据误判为合法日期。

替换表达式按“月-日-年”重排为 `\3-\4-\1`。

In [7]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
date_pattern = r"(\d{4})([-/.])(\d{2})\2(\d{2})"

print("== 转换前 ==")
print(dates)
print(f"（共 {len(dates.splitlines())} 行）")

new_dates = re.sub(date_pattern, r"\3-\4-\1", dates)

print("\n== 转换后（月-日-年） ==")
print(new_dates)
print(f"（共 {len(new_dates.splitlines())} 行，行数保持不变）")

print("\n逐行核对：")
for before, after in zip(dates.splitlines(), new_dates.splitlines()):
    print(f"  {before}  ->  {after}")

== 转换前 ==
2024-05-01
2024/05/02
2024.05.03
（共 3 行）

== 转换后（月-日-年） ==
05-01-2024
05-02-2024
05-03-2024
（共 3 行，行数保持不变）

逐行核对：
  2024-05-01  ->  05-01-2024
  2024/05/02  ->  05-02-2024
  2024.05.03  ->  05-03-2024


**结果解释（任务 3）**：`re.sub` 默认替换**所有**匹配，三行分别变成 `05-01-2024`、`05-02-2024`、`05-03-2024`。`\1`~`\4` 依次是年、分隔符、月、日，替换串 `\3-\4-\1` 先放月、再放日、最后放年，因此格式改变而分隔符统一成 `-`，行数也不变。

**任务 4：电话号码统一为 `(123) 456-7890`**

原文本里两种写法混用（`123-456-7890` 与 `987.654.3210`），可以用 `(\d{3})[.-](\d{3})[.-](\d{4})` 一次统一：

- 三个捕获组分别是区号、局号、后四位；
- 字符类 `[.-]` 同时接受 `-` 和 `.` 两种分隔；
- 替换串 `(\1) \2-\3` 把区号加括号、加上空格，再拼成 `456-7890` 形式。

In [8]:
phones = "联系我：123-456-7890 或 987.654.3210"
phone_pattern = r"(\d{3})[.-](\d{3})[.-](\d{4})"

print("原文：", phones)
fixed = re.sub(phone_pattern, r"(\1) \2-\3", phones)
print("统一后：", fixed)

print("\n逐个号码核对：")
for m in re.finditer(phone_pattern, phones):
    print(f"  {m.group(0)!r}  ->  ({m.group(1)}) {m.group(2)}-{m.group(3)}")

原文： 联系我：123-456-7890 或 987.654.3210
统一后： 联系我：(123) 456-7890 或 (987) 654-3210

逐个号码核对：
  '123-456-7890'  ->  (123) 456-7890
  '987.654.3210'  ->  (987) 654-3210


**结果解释（任务 4）**：两处号码都被改写为 `(123) 456-7890` 与 `(987) 654-3210`，括号和空格由替换串直接生成；因为 `[.-]` 是字符类，点号在类内是字面量，不需要转义。

### 实验3：使用 BeautifulSoup 解析 HTML

`BeautifulSoup(html_doc, "html.parser")` 把 HTML 字符串解析成**树形结构**，之后可以用“标签名 + 属性”定位节点，比正则可靠得多（标签的位置、嵌套、空白都不影响查找）。

**任务 1：构建解析树，取标题与 h1**

In [9]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

print("解析树对象类型：", type(soup).__name__)
print("页面标题文本：", soup.title.get_text())
print("h1 的标签名：", soup.h1.name)
print("h1 的文本  ：", soup.h1.get_text())
print("\n美化后的解析树：")
print(soup.prettify())

解析树对象类型： BeautifulSoup
页面标题文本： 实验课程
h1 的标签名： h1
h1 的文本  ： 数据分析

美化后的解析树：
<html>
 <head>
  <title>
   实验课程
  </title>
 </head>
 <body>
  <h1>
   数据分析
  </h1>
  <p class="course">
   第六讲：正则表达式
  </p>
  <p class="course">
   第七讲：结构化数据
  </p>
  <a href="https://example.org/lecture06">
   讲义六
  </a>
  <a href="https://example.org/lecture07">
   讲义七
  </a>
 </body>
</html>



**结果解释（任务 1）**：`soup.title` 返回第一个 `<title>` 标签对象，`get_text()` 取其中的纯文本 `实验课程`；`soup.h1.name` 是字符串 `'h1'`（标签名，不含尖括号），`soup.h1.get_text()` 是 `数据分析`。`prettify()` 只是把树按缩进重新排版，用于观察层级关系。

**任务 2：用 `find_all()` 定位段落与链接**

- `soup.find_all("p", class_="course")`：标签名 `p` + 属性筛选；`class` 是 Python 关键字，所以写作 `class_`。
- `soup.find_all("a")`：取出所有链接，再用 `a.get("href")` 读属性、`a.get_text()` 读链接文字。

In [10]:
courses = soup.find_all("p", class_="course")
print(f'class="course" 的段落共 {len(courses)} 个：')
for i, p in enumerate(courses, 1):
    print(f"  {i}. 标签={p.name}，文字={p.get_text()}")

links = soup.find_all("a")
print(f"\n链接共 {len(links)} 个：")
for i, a in enumerate(links, 1):
    print(f"  {i}. 链接文字 = {a.get_text()}，href = {a.get('href')}")

class="course" 的段落共 2 个：
  1. 标签=p，文字=第六讲：正则表达式
  2. 标签=p，文字=第七讲：结构化数据

链接共 2 个：
  1. 链接文字 = 讲义六，href = https://example.org/lecture06
  2. 链接文字 = 讲义七，href = https://example.org/lecture07


**结果解释（任务 2）**：两条课程段落都被 `class="course"` 选出，说明按属性筛选有效；两个链接的 `href` 原样取出，分别是 `lecture06` 与 `lecture07`，链接文字为 `讲义六`、`讲义七`。如果没有 `class` 属性，`find_all("p", class_="course")` 会返回空列表。

**任务 3：`get_text()` 层次比较**

对整体文档、单个段落、单个链接分别调用 `get_text()`，观察提取出的文本范围与空白差异。

In [11]:
print("== 1) 整个文档 ==")
whole = soup.get_text()
print(repr(whole))

print("\n== 2) 单个段落（第一个 course 段落） ==")
one_p = soup.find("p", class_="course").get_text()
print(repr(one_p))

print("\n== 3) 单个链接（第一个 <a>） ==")
one_a = soup.find("a").get_text()
print(repr(one_a))

print("\n== 对比 ==")
print(f"整页文本长度 = {len(whole)}，单段落长度 = {len(one_p)}，单链接长度 = {len(one_a)}")
print("整页 get_text() 会把所有标签文字用换行拼起来，并保留标签间的空白：")
print(whole)

print("\n用 stripped_strings 去掉空行后再拼接，得到更干净的可见文本：")
print(" | ".join(soup.stripped_strings))

== 1) 整个文档 ==
'\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'

== 2) 单个段落（第一个 course 段落） ==
'第六讲：正则表达式'

== 3) 单个链接（第一个 <a>） ==
'讲义六'

== 对比 ==
整页文本长度 = 40，单段落长度 = 9，单链接长度 = 3
整页 get_text() 会把所有标签文字用换行拼起来，并保留标签间的空白：

实验课程
数据分析
第六讲：正则表达式
第七讲：结构化数据
讲义六
讲义七



用 stripped_strings 去掉空行后再拼接，得到更干净的可见文本：
实验课程 | 数据分析 | 第六讲：正则表达式 | 第七讲：结构化数据 | 讲义六 | 讲义七


**结果解释（任务 3）**

| 对象 | `get_text()` 结果 | 特点 |
| --- | --- | --- |
| 整个文档 | 含 `实验课程`、`数据分析`、两条课程段落、两个链接文字 | 把树里**所有**可见文字按文档顺序拼起来，标签之间的换行/缩进作为空白一并保留，因此出现多个空行 |
| 单个段落 | `第六讲：正则表达式` | 作用范围仅该节点及其子孙，段落内部没有多余空白，得到“干净”的一句话 |
| 单个链接 | `讲义六` | 只返回 `<a>` 内的文字，不含 `href` 属性值 |

结论：`get_text()` 的范围取决于调用它的节点；`stripped_strings` 会逐段去掉首尾空白，适合做正文抽取。属性（如 `href`）属于标签信息，不在可见文本里，必须用 `get("属性名")` 单独读取。

## 三、实验小结

1. **正则的基础元素**：`\d`/`\s`/`\w` 是字符类，`\b` 是零宽的单词边界断言，`*`、`+`、`?`、`{m,n}` 控制重复次数；`\bcat\b` 只匹配独立单词，这是它与 `cat` 的关键区别。
2. **交替符 `|` 有序**：`a|aa|aaa` 与 `aaa|aa|a` 在同一字符串上结果不同，写正则时应把更长、更具体的分支放在前面；默认区分大小写，需要时用 `re.IGNORECASE`。
3. **捕获组与后向引用**：`(...)` 编号从 1 开始，`findall` 在有多组时返回元组；`\1` 引用前面捕获的内容，可用来检测连续重复单词；`re.sub` 的替换串同样能引用 `\1`、`\2`… 来重排、格式化文本（日期、电话）。
4. **结构化解析优于字符串切分**：BeautifulSoup 把 HTML 变成树，`find_all` 按“标签名 + 属性”精确定位，`get_text()` 只取可见文本，属性需单独读取；相比正则，它对嵌套、空白和标签顺序都不敏感。
5. **环境规范**：全部单元在独立 conda 环境 `exp03`（Python 3.12）中重启内核后 Run All 通过，输出可复现，没有依赖旧变量。